In [7]:
# import shutup
# shutup.please()

import warnings
warnings.filterwarnings('ignore')

# import rootutils
# ROOT = rootutils.setup_root(indicator='README.md', search_from=os.path.abspath(''), pythonpath=True, cwd=True)

import jax
import jax.numpy as jnp
import optax
from jaxtyping import ArrayLike, Float
import numpy as np
GLOBAL_KEY = jax.random.key(42)

import seaborn as sns
import matplotlib.pyplot as plt
# plt.style.use(['science', 'notebook'])

# Lagrangian Potentials

from ott2.neural.methods.lagrangian.lagrangian_potentials import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from torch.utils.data import Dataset, DataLoader

class InfiniteLoaderWrapper:
    def __init__(self, loader: DataLoader):
        self.loader = loader
        self.loader_it = iter(loader)
    
    def __iter__(self):
        self.loader_it = iter(self.loader)
        return self

    def __next__(self):
        try:
            batch = next(self.loader_it)
        except StopIteration:
            self.loader_it = iter(self.loader)
            batch = next(self.loader_it)
        return batch

class OTLoader:
    def __init__(
        self,
        src_ds: Dataset,
        trg_ds: Dataset,
        flatten_flag: bool = False,
        **torch_dataloader_kwargs,
    ):
        def collate_fn(batch: tuple[np.ndarray]):
            return np.stack(batch)

        self.src_loader = InfiniteLoaderWrapper(DataLoader(src_ds, collate_fn=collate_fn, **torch_dataloader_kwargs))
        self.trg_loader = InfiniteLoaderWrapper(DataLoader(trg_ds, collate_fn=collate_fn, **torch_dataloader_kwargs))
        self.flatten_flag = flatten_flag

    def __iter__(self):
        self.src_loader = iter(self.src_loader)
        self.trg_loader = iter(self.trg_loader)
        return self

    def __next__(self):
        src_batch = jnp.asarray(next(self.src_loader))
        tgt_batch = jnp.asarray(next(self.trg_loader))
        if self.flatten_flag:
            b_size = src_batch.shape[0]
            src_batch = src_batch.reshape(b_size, -1)
            tgt_batch = tgt_batch.reshape(b_size, -1)
        return {
            "src_lin": src_batch,
            "tgt_lin": tgt_batch,
        }

In [10]:
from torch.utils.data import Subset, DataLoader, Dataset, ConcatDataset
from torchvision.transforms import Compose, Resize, Normalize, ToTensor, RandomCrop, RandomHorizontalFlip, RandomVerticalFlip, Lambda, Pad, CenterCrop, RandomResizedCrop
from torchvision.datasets import ImageFolder

class MyImageFolder(ImageFolder):
    def __getitem__(self, idx: int):
        return super().__getitem__(idx)[0].numpy()

img_size = 64

## anime dataset
path = "/home/jovyan/nazar/aligned_anime_faces"
transform = Compose([Resize((img_size, img_size)), ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
anime_dataset = MyImageFolder(path, transform=transform)

## celeba female dataset
path = "/home/jovyan/nazar/celeba_female"
attrs_path = "/home/jovyan/nazar/list_attr_celeba.txt" 
transform = Compose([Resize((img_size, img_size)), ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
celeba_female_dataset = MyImageFolder(path, transform=transform)

In [13]:
batch_size = 64
anime_loader = DataLoader(
    anime_dataset,
    shuffle=False,
    batch_size=batch_size,
    num_workers=4,
)

In [14]:
import flax
import tools.jax_inception as inception
from tools.fid import get_loader_stats, get_pushed_loader_stats, calculate_frechet_distance

In [17]:
inception_net = inception.InceptionV3(pretrained=True)
rng = jax.random.PRNGKey(0)
inception_params = inception_net.init(rng, jnp.ones((1, 299, 299, 3)))
inception_apply = jax.jit(functools.partial(inception_net.apply, train=False))

In [20]:
mu_data, sigma_data = get_loader_stats(anime_loader, inception_apply, inception_params, batch_size=128, n_epochs=1, verbose=True, classes=False)

5993it [03:55, 25.50it/s]


KeyboardInterrupt: 